# Ajuste fino de texto: ¿cuándo le gana a los embeddings?

**Sesión 7 · IA Intermedio · EstudIA 2026**

En la sesión 6 el transformer quedó congelado: convertimos cada texto en un vector y solo entrenamos una regresión
logística encima. Hoy lo **ajustamos**: entrenamos también el transformer con nuestros ejemplos, con el `Trainer` de
Hugging Face, la misma receta de la sesión 4, ahora con texto.

**El recorrido.**

| Parte | Qué hacemos |
|---|---|
| 1 | Lo que recibe el modelo |
| 2 | Los tuits, por tercera vez: ajustar todo el modelo |
| 3 | Tres maneras de ajustar |
| 4 | Tu modelo en un `pipeline` |
| 5 | ¿Cuándo vale la pena ajustar? |
| 6 | El registro de experimentos |

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. Activa una GPU si puedes (*Entorno de ejecución ›
Cambiar tipo de entorno de ejecución › GPU T4*). Sin GPU también funciona, pero cada entrenamiento tarda varios
minutos: puedes seguir la demostración en la pantalla y ejecutar solo las partes 1, 5 y 6.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, json, urllib.request, warnings
warnings.filterwarnings("ignore")
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s07")
TUITS = "https://huggingface.co/api/datasets/cardiffnlp/tweet_sentiment_multilingual/parquet/spanish/{}/0.parquet"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments,
                          DataCollatorWithPadding, set_seed, pipeline)
from transformers.utils import logging as registro_hf
registro_hf.set_verbosity_error()   # sin avisos largos al cargar los modelos
try:
    from peft import LoraConfig, get_peft_model
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
    from peft import LoraConfig, get_peft_model

GPU = torch.cuda.is_available()
torch.set_num_threads(max(1, os.cpu_count() or 1))
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: cada entrenamiento tarda varios minutos.")

In [ ]:
# @title Funciones auxiliares: descargar, escribir números, ajustar, predecir, dibujar y registrar
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

class Ajuste:
    """Lo que queda de un ajuste fino: el entrenador, el tokenizador, las clases y unos datos para el registro."""
    def __init__(self, **datos):
        self.__dict__.update(datos)

def _ejemplos(tok, textos, largo, clases=None, etiquetas=None):
    """Convierte los textos en tokens (cortando en `largo`) y, si hay etiquetas, les pone su número de clase."""
    cod = tok([str(t) for t in textos], truncation=True, max_length=largo)
    filas = [{k: cod[k][i] for k in cod} for i in range(len(textos))]
    if etiquetas is not None:
        for fila, e in zip(filas, etiquetas):
            fila["labels"] = clases.index(str(e))
    return filas

def ajustar(textos, etiquetas, modelo, forma="todo", tasa=5e-5, epocas=3, lote=16, largo=128,
            val_textos=None, val_etiquetas=None, semilla=0, mostrar=True):
    """Ajuste fino de un modelo de texto de Hugging Face para clasificar, con el Trainer.

    forma: "todo" (todo el modelo), "cabeza" (solo la capa nueva de encima) o "lora" (matrices pequeñas añadidas).
    Si le das textos de validación, mide la exactitud de validación al final de cada época."""
    set_seed(semilla)
    clases = sorted({str(e) for e in etiquetas})
    tok = AutoTokenizer.from_pretrained(modelo)
    red = AutoModelForSequenceClassification.from_pretrained(
        modelo, num_labels=len(clases), id2label=dict(enumerate(clases)), label2id={c: i for i, c in enumerate(clases)})
    if forma == "cabeza":
        for nombre, p in red.named_parameters():
            p.requires_grad = nombre.startswith("classifier")
    elif forma == "lora":
        red = get_peft_model(red, LoraConfig(task_type="SEQ_CLS", r=16, lora_alpha=32, lora_dropout=0.1))
    entrenables = sum(p.numel() for p in red.parameters() if p.requires_grad)
    total = sum(p.numel() for p in red.parameters())
    hay_val = val_textos is not None
    args = TrainingArguments(output_dir="entrenamiento", per_device_train_batch_size=lote,
                             per_device_eval_batch_size=64, learning_rate=tasa, num_train_epochs=epocas,
                             warmup_steps=0.1, weight_decay=0.01, eval_strategy="epoch" if hay_val else "no",
                             logging_strategy="epoch", save_strategy="no", report_to="none", seed=semilla,
                             fp16=torch.cuda.is_available(), disable_tqdm=not mostrar)
    entrenador = Trainer(
        model=red, args=args, data_collator=DataCollatorWithPadding(tok),
        train_dataset=_ejemplos(tok, textos, largo, clases, etiquetas),
        eval_dataset=_ejemplos(tok, val_textos, largo, clases, val_etiquetas) if hay_val else None,
        compute_metrics=lambda p: {"exactitud": float((p.predictions.argmax(-1) == p.label_ids).mean())})
    inicio = time.time()
    entrenador.train()
    return Ajuste(entrenador=entrenador, tok=tok, clases=clases, largo=largo, segundos=time.time() - inicio,
                  entrenables=entrenables, total=total, modelo=modelo, forma=forma)

def predecir(ajuste, textos):
    """La clase que el modelo ajustado le da a cada texto."""
    salida = ajuste.entrenador.predict(_ejemplos(ajuste.tok, textos, ajuste.largo))
    return np.array(ajuste.clases)[salida.predictions.argmax(-1)]

def dibujar_historia(ajuste, titulo="Cómo fue el entrenamiento"):
    """Las curvas de la sesión 2: pérdida de entrenamiento y de validación, y exactitud de validación, por época."""
    h = ajuste.entrenador.state.log_history
    ent = [(r["epoch"], r["loss"]) for r in h if "loss" in r]
    val = [(r["epoch"], r["eval_loss"], r["eval_exactitud"]) for r in h if "eval_loss" in r]
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3))
    a.plot([e for e, _ in ent], [l for _, l in ent], "o-", color="#808080", label="Entrenamiento")
    if val:
        a.plot([v[0] for v in val], [v[1] for v in val], "o-", color="#CB318E", label="Validación")
        b.plot([v[0] for v in val], [v[2] for v in val], "o-", color="#CB318E")
        for e, _, x in val:
            b.annotate(pct(x), (e, x), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=9)
        b.set_ylabel("Exactitud de validación"); b.set_ylim(min(v[2] for v in val) - 0.1, 1)
    a.set_ylabel("Pérdida"); a.legend(frameon=False)
    for eje in (a, b):
        eje.set_xlabel("Época"); eje.spines[["top", "right"]].set_visible(False); eje.grid(alpha=0.3)
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

def millones(n):
    """118_000_000 -> "118 millones"; 12_345 -> "12.345"."""
    return f"{n / 1e6:.1f} millones".replace(".0 ", " ").replace(".", ",") if n >= 1e6 else miles(n)

from IPython.display import display
REGISTRO = "experimentos.csv"

def registrar(**datos):
    """Agrega una fila a tu registro de experimentos (el archivo experimentos.csv): qué probaste y qué salió."""
    fila = pd.DataFrame([{"hora": time.strftime("%H:%M"), **datos}])
    tabla = pd.concat([pd.read_csv(REGISTRO), fila], ignore_index=True) if os.path.exists(REGISTRO) else fila
    tabla.to_csv(REGISTRO, index=False)

def ver_registro():
    """Muestra el registro de experimentos, el más reciente abajo."""
    if not os.path.exists(REGISTRO):
        print("Todavía no hay experimentos en el registro.")
        return
    tabla = pd.read_csv(REGISTRO)
    display(tabla.style.format(precision=1, decimal=",", thousands=".", na_rep="").hide(axis="index"))

---
## 1. Lo que recibe el modelo

Los mismos tuits en español de las sesiones 5 y 6 (negativo, neutral, positivo). Antes de entrenar, el **tokenizador**
del modelo parte cada texto en pedazos (tokens) y le da un número a cada uno. Tres detalles que en las imágenes no
existían:

- **Cortar** (`truncation`): los textos largos se cortan en `largo` tokens.
- **Rellenar** (`padding`): en cada lote, los textos cortos se completan con un token de relleno hasta igualar al más
  largo del lote. La **máscara de atención** le dice al modelo cuáles tokens son relleno.
- **Rellenar por lote**: igualar cada lote a su texto más largo, y no a un largo fijo, ahorra mucho cálculo.

In [ ]:
NOMBRES = ["negativo", "neutral", "positivo"]
ent, val, prueba = (pd.read_parquet(descargar(f"tuits_{p}.parquet", TUITS.format(p)))
                    for p in ("train", "validation", "test"))
for df in (ent, val, prueba):
    df["sentimiento"] = [NOMBRES[i] for i in df.label]
print(f"Tuits: {miles(len(ent))} de entrenamiento, {miles(len(val))} de validación, {miles(len(prueba))} de prueba.")

MODELO = "intfloat/multilingual-e5-small"
tok = AutoTokenizer.from_pretrained(MODELO)
ejemplo = "No me gustó para nada la película, parcero."
cod = tok(ejemplo)
print("\nLa frase:", ejemplo)
print("Sus tokens:", tok.convert_ids_to_tokens(cod["input_ids"]))
print("Sus números:", cod["input_ids"])

largos = [len(ids) for ids in tok(list(ent.text))["input_ids"]]
print(f"\nTokens por tuit: mediana {int(np.median(largos))}, el más largo {max(largos)}.",
      f"Con largo = 128 no se corta ninguno." if max(largos) <= 128 else "")

In [ ]:
# Un lote de tres tuits: se rellenan hasta el más largo, y la máscara marca el relleno con 0.
lote = DataCollatorWithPadding(tok)([tok(t) for t in ent.text.iloc[:3]])
print("Forma del lote:", tuple(lote["input_ids"].shape), "(3 tuits × tokens del más largo)")
print("Máscara de atención:")
print(lote["attention_mask"].numpy())

**Para pensar.**

- ¿Qué palabras quedaron partidas en varios tokens? ¿Por qué crees que el tokenizador las parte?

---
## 2. Los tuits, por tercera vez: ajustar todo el modelo

`AutoModelForSequenceClassification` carga el transformer y le pone encima una **capa nueva** con una salida por clase.
Entrenamos todo: la capa nueva desde cero y el transformer un poco, con una **tasa de aprendizaje pequeña** para no
borrar lo que ya sabía. Pocas épocas bastan.

Le damos los tuits de validación para ver las curvas de la sesión 2 al final de cada época.

In [ ]:
todo = ajustar(ent.text, ent.sentimiento, MODELO, forma="todo", tasa=5e-5, epocas=3,
               val_textos=val.text, val_etiquetas=val.sentimiento)
pred_todo = predecir(todo, prueba.text)
EXACTITUD_TODO = (pred_todo == prueba.sentimiento.to_numpy()).mean()
print(f"Entrenado en {todo.segundos:.0f} s. Exactitud en prueba: {pct(EXACTITUD_TODO)}")
registrar(experimento="Tuits: ajuste fino, todo el modelo", modelo=MODELO, tasa=5e-5, epocas=3,
          segundos=round(todo.segundos), exactitud_prueba=round(100 * EXACTITUD_TODO, 1))
dibujar_historia(todo)

In [ ]:
filas = [("Bolsa de palabras (sesión 5)", 0.5230),
         ("GRU, vectores de fastText (sesión 5)", 0.5356),
         ("Embeddings congelados + regresión (sesión 6)", 0.6138),
         ("Ajuste fino de e5, todo el modelo (hoy)", EXACTITUD_TODO)]
fig, eje = plt.subplots(figsize=(8, 2.6))
eje.barh(range(4), [f[1] for f in filas], color=["#B0B0B0", "#B0B0B0", "#000000", "#CB318E"])
for i, f in enumerate(filas):
    eje.text(f[1] + 0.01, i, pct(f[1]), va="center")
eje.axvline(1 / 3, color="#7FC200", ls="--", lw=1.5, label="Responder siempre lo mismo (33 %)")
eje.set_yticks(range(4), [f[0] for f in filas]); eje.invert_yaxis(); eje.set_xlim(0, 1)
eje.set_xlabel(f"Exactitud con los {miles(len(prueba))} tuits de prueba"); eje.legend(frameon=False, loc="lower right")
eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()

**Para pensar.**

- ¿Las curvas muestran sobreajuste? ¿En qué época lo verías?
- Con los mismos 1.839 tuits, ¿qué cambió entre la sesión 6 y hoy?

---
## 3. Tres maneras de ajustar

| Forma | Qué se entrena |
|---|---|
| **Solo la cabeza** | La capa nueva de encima; el transformer queda congelado. Se parece a la sesión 6. |
| **LoRA** | El transformer queda congelado, pero le añadimos **matrices pequeñas** al lado de algunas capas y entrenamos solo esas (y la cabeza). |
| **Todo el modelo** | Todo, como en la parte 2. |

Con LoRA, lo que hay que guardar de tu ajuste pesa unos pocos megabytes y no cientos. En modelos grandes, además, cabe
en una GPU pequeña. Cada forma necesita su propia tasa de aprendizaje y sus épocas: mientras menos se entrena, más
grande puede ser la tasa.

Esta vez usamos **BETO**, el modelo en español de la sesión 6. e5 está hecho para leerse con el promedio de todos sus
tokens, y la cabeza de clasificación lee solo el primero; BETO sí entrenó ese primer token para resumir la frase, así
que con él la comparación es justa.

In [ ]:
BETO = "dccuchile/bert-base-spanish-wwm-cased"
recetas = [("Solo la cabeza", "cabeza", 5e-3, 10), ("LoRA", "lora", 1e-3, 5), ("Todo el modelo", "todo", 3e-5, 3)]
resumen, ajustes_beto = [], {}
for nombre, forma, tasa, epocas in recetas:
    a = ajustar(ent.text, ent.sentimiento, BETO, forma=forma, tasa=tasa, epocas=epocas, mostrar=False)
    ajustes_beto[forma] = a
    exactitud = (predecir(a, prueba.text) == prueba.sentimiento.to_numpy()).mean()
    resumen.append({"Forma (BETO)": nombre, "Parámetros que se entrenan": millones(a.entrenables),
                    "Del total": pct(a.entrenables / a.total), "Épocas": epocas, "Segundos": round(a.segundos),
                    "Exactitud en prueba": pct(exactitud)})
    registrar(experimento=f"Tuits: BETO, {nombre.lower()}", modelo=BETO, tasa=tasa, epocas=epocas,
              segundos=round(a.segundos), exactitud_prueba=round(100 * exactitud, 1))
    print(f"{nombre}: listo en {a.segundos:.0f} s.")
resumen.append({"Forma (BETO)": "(e5-small, todo el modelo, parte 2)",
                "Parámetros que se entrenan": millones(todo.entrenables), "Del total": "100,0 %", "Épocas": 3,
                "Segundos": round(todo.segundos), "Exactitud en prueba": pct(EXACTITUD_TODO)})
display(pd.DataFrame(resumen).style.hide(axis="index"))

**Para pensar.**

- ¿Cuál forma da más por cada parámetro que se entrena? ¿Cuál usarías con un modelo diez veces más grande?
- BETO ajustado le gana a e5 ajustado con los mismos tuits. ¿Qué tiene BETO que e5 no tiene?

---
## 4. Tu modelo en un `pipeline`

Guardamos el mejor modelo, BETO con todo el modelo ajustado, en una carpeta y lo cargamos con el mismo `pipeline` de
la sesión 1. Ahora el modelo de la ficha es el tuyo. Y volvemos a los pares de la negación.

In [ ]:
mejor = ajustes_beto["todo"]
mejor.entrenador.save_model("mi_modelo_tuits")
mejor.tok.save_pretrained("mi_modelo_tuits")
clasificador = pipeline("text-classification", model="mi_modelo_tuits", device=0 if GPU else -1)
for frase in ['La atención en el centro de salud fue excelente, me resolvieron todo.', 'Llevo una hora esperando el bus y nada.', 'Mañana empiezan las clases a las siete.']:
    r = clasificador(frase)[0]
    print(f"{r['label']:9s} {pct(r['score'], 0):>5s}  {frase}")

print("\nLos pares de la negación:")
for a, b in [('no fue malo, fue bueno', 'no fue bueno, fue malo'), ('me gustó, no me aburrí', 'me aburrí, no me gustó')]:
    for frase in (a, b):
        r = clasificador(frase)[0]
        print(f"{r['label']:9s} {pct(r['score'], 0):>5s}  {frase}")
tamano = sum(os.path.getsize(os.path.join("mi_modelo_tuits", f)) for f in os.listdir("mi_modelo_tuits"))
print(f"\nLa carpeta del modelo pesa {tamano / 1e6:.0f} MB.".replace(".", ","))

**Para pensar.**

- ¿Acierta los pares de la negación? En la sesión 6, los embeddings congelados acertaron 3 de 4.
- Si quisieras compartir este modelo con tu equipo, ¿qué escribirías en su ficha?

---
## 5. ¿Cuándo vale la pena ajustar?

La pregunta de hoy. Volvemos a la curva de la sesión 6 con lo que la gente le pide a un asistente de voz (60
intenciones): bolsa de palabras y embeddings congelados con *k* ejemplos por intención, y ahora también el ajuste fino
de e5 con los mismos ejemplos. Cada punto es el promedio de tres sorteos. Entrenar 24 modelos toma tiempo, así que
estos resultados ya vienen calculados (en una GPU, con la función `ajustar` de este notebook).

In [ ]:
curva = json.load(open(descargar("curva_asistente.json")))
KS = curva["ks"]
fig, eje = plt.subplots(figsize=(8, 3.6))
eje.plot(KS, curva["bolsa"], "o-", color="#B0B0B0", lw=2, label="Bolsa de palabras")
eje.plot(KS, curva["congelados"], "o-", color="#000000", lw=2, label="Embeddings congelados + regresión (sesión 6)")
eje.plot(KS, curva["ajuste_fino"], "o-", color="#CB318E", lw=2, label="Ajuste fino de e5 (hoy)")
eje.set_xscale("log", base=2); eje.set_xticks(KS, [str(k) for k in KS]); eje.set_ylim(0, 1)
eje.set_xlabel("Ejemplos por intención"); eje.set_ylabel("Exactitud en prueba"); eje.grid(alpha=0.3)
eje.legend(frameon=False, loc="lower right"); eje.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.show()
for k, b, c, a in zip(KS, curva["bolsa"], curva["congelados"], curva["ajuste_fino"]):
    print(f"k = {k:2d}: bolsa {pct(b)} · congelados {pct(c)} · ajuste fino {pct(a)}")
print(f"Con todos los ejemplos ({miles(curva['n_todo'])}): bolsa {pct(curva['bolsa_todo'])} · "
      f"congelados {pct(curva['congelados_todo'])} · ajuste fino {pct(curva['ajuste_fino_todo'])}")

**Para pensar.**

- ¿Desde cuántos ejemplos por intención le gana el ajuste fino a los embeddings congelados?
- Tu proyecto tiene de 20 a 50 ejemplos en total. ¿Qué te dice esta curva?

---
## 6. El registro de experimentos

Cada entrenamiento de hoy quedó en una fila de `experimentos.csv`: qué modelo, qué tasa, cuántas épocas, cuánto tardó y
qué salió. Sin registro, a la tercera prueba ya no recuerdas qué cambiaste.

Herramientas como **Weights & Biases** o **MLflow** hacen lo mismo a lo grande: guardan cada entrenamiento con sus
curvas, lo comparan y lo comparten con tu equipo. Para un proyecto como el tuyo, un archivo como este alcanza.

In [ ]:
ver_registro()

**Lo que viene.** En la competencia de hoy volvemos a los contratos de la sesión 6, con más ejemplos etiquetados. En
la sesión 8 vemos modelos que entienden imagen y texto a la vez.

**Fuentes.** Tuits: Barbieri, F., Espinosa Anke, L. y Camacho-Collados, J. (2022). XLM-T. *LREC*.
`cardiffnlp/tweet_sentiment_multilingual` (español), CC BY 3.0 y términos de Twitter, para uso académico. Textos del
asistente: FitzGerald, J. y otros (2023). MASSIVE. *ACL*. CC BY 4.0. e5: Wang, L. y otros (2024). Multilingual E5 Text
Embeddings: A Technical Report. Licencia MIT. LoRA: Hu, E. y otros (2022). LoRA: Low-Rank Adaptation of Large Language
Models. *ICLR*.